In [1]:
import json
from typing import Dict, Any
from invoke_agent.core import execute_api_call
from invoke_agent.context import load_agents_map

AGENTS_TO_TEST = [
    "open-meteo",
    "open-weather-map",
    "google-directions",
    "google-places",
    "google-calendar", # 2 failures - bad examples
    "google-gmail-send", # 4 failures - Bad examples
    "google-tasks", # ???
    "google-contacts", # ???
    "google-drive", # ???
    "youtube", # 2 failures - bad examples
    "microsoft-outlook", # 2 failures - Bad examples
    "microsoft-calendar", # 2 failures - Bad examples
    "microsoft-onenote",
    "notion", # 4 failures - no premium
    "calendly", # No account
]

failures = []

def extract_examples_from_agents_map(name: str, rendered_txt: str) -> list[Dict[str, Any]]:
    """
    Extract example blocks from rendered agents.txt.
    """
    examples = []
    lines = rendered_txt.splitlines()
    for i, line in enumerate(lines):
        if line.strip() == '```json':
            block = []
            for j in range(i + 1, len(lines)):
                if lines[j].strip() == '```':
                    break
                block.append(lines[j])
            try:
                example = json.loads("\n".join(block))
                examples.append(example)
            except Exception as e:
                failures.append((name, f"Invalid JSON in example block: {e}"))
    return examples

def execute_example(example: Dict[str, Any]) -> Any:
    payload = {
        "action": example["method"],
        "url": example["url"],
    }
    if "auth_code" in example:
        payload["auth_code"] = example["auth_code"]
    if "headers" in example:
        payload["headers"] = example["headers"]
    if "parameters" in example:
        payload["parameters"] = example["parameters"]

    return execute_api_call(payload)

if __name__ == "__main__":
    agents_map = load_agents_map(AGENTS_TO_TEST)

    for name, rendered_txt in agents_map.items():
        print(f"\n🔍 Testing agent: {name}")
        examples = extract_examples_from_agents_map(name, rendered_txt)
        for i, example in enumerate(examples):
            try:
                output = execute_example(example)
                if isinstance(output, dict):
                    if output.get("success") is False:
                        msg = output.get("message", "Unsuccessful response")
                        print(f"  ❌ Example {i+1} responded with success=False: {msg}")
                        failures.append((f"{name} [example {i+1}]", msg))
                    elif "error" in output:
                        print(f"  ❌ Example {i+1} responded with error: {output['error']}")
                        failures.append((f"{name} [example {i+1}]", output["error"]))
                    else:
                        print(f"  ✅ Example {i+1} succeeded.")
                else:
                    print(f"  ⚠️ Example {i+1} returned non-dict output: {output}")
                    failures.append((f"{name} [example {i+1}]", "Non-dict response"))
            except Exception as e:
                print(f"  ❌ Example {i+1} raised an exception: {e}")
                failures.append((f"{name} [example {i+1}]", str(e)))

    if failures:
        print(f"\n🚨 {len(failures)} failures:")
        for name, err in failures:
            print(f"- {name}: {err}")
        exit(1)

    print("\n✅ All tests complete.")


🔍 Testing agent: open-meteo
  ✅ Example 1 succeeded.
  ✅ Example 2 succeeded.

🔍 Testing agent: open-weather-map
  ✅ Example 1 succeeded.
  ✅ Example 2 succeeded.
  ✅ Example 3 succeeded.

🔍 Testing agent: google-directions
  ✅ Example 1 succeeded.
  ✅ Example 2 succeeded.

🔍 Testing agent: google-places
  ✅ Example 1 succeeded.
  ✅ Example 2 succeeded.
  ✅ Example 3 succeeded.
  ✅ Example 4 succeeded.
  ✅ Example 5 succeeded.

🔍 Testing agent: google-calendar
  ❌ Example 1 responded with success=False: Authentication required
  ❌ Example 2 responded with success=False: Authentication required
  ❌ Example 3 responded with success=False: Authentication required
  ❌ Example 4 responded with success=False: Authentication required
  ❌ Example 5 responded with success=False: Authentication required

🔍 Testing agent: google-gmail-send

🔍 Testing agent: google-tasks
  ❌ Example 1 responded with success=False: Authentication required



KeyboardInterrupt

